# Figure 2 — Discrimination of consciousness by higher-order interactions

This notebook generates **Figure 2** of the paper:
- **Left panels**: violin plots of $\Omega$ distributions across conditions for the optimal
  n-plets ($\Omega_C > \Omega_{NR}$ and $\Omega_{NR} > \Omega_C$ polarities).
- **Right panels**: regional participation brain maps from the high-performing n-plet tails
  (output of `R1_B_region_sampling.ipynb`).

**Inputs** (from `results/`):
- `covariance_matrices_gc.h5` — per-scan covariance matrices
- `R1_C_region_maps_PRAUC_deltaO.pkl.gz` — pre-computed region participation maps

**Optimal n-plets** (from `R1_B_nplet_eval_*.csv`):

| Dataset | Polarity | Order | n-plet |
|---------|----------|-------|--------|
| MA | $\Omega_C > \Omega_{NR}$ | 7 | `[7, 14, 23, 26, 35, 55, 67]` |
| MA | $\Omega_{NR} > \Omega_C$ | 4 | `[38, 40, 45, 81]` |
| DBS | $\Omega_C > \Omega_{NR}$ | 9 | `[6, 8, 12, 18, 20, 22, 59, 61, 63]` |
| DBS | $\Omega_{NR} > \Omega_C$ | 3 | `[3, 4, 10]` |

**Note**: Run from the `high-order-anesthesia` project root.

In [ ]:
from pathlib import Path
import os

def ensure_project_root(target_name: str = "high-order-anesthesia") -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name == target_name:
        return cwd
    for parent in cwd.parents:
        if parent.name == target_name:
            os.chdir(parent)
            return parent
    raise RuntimeError(f"Could not find '{target_name}' in path.")

ROOT = ensure_project_root()
print(f"Now in: {ROOT.name}")

In [ ]:
import torch
import numpy as np
import pandas as pd
import pickle
import gzip
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.stats import mannwhitneyu
from thoi.measures.gaussian_copula import nplets_measures
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [ ]:
from src.hoi_anesthesia.io import load_covariance_dict
from src.hoi_anesthesia.plotting import plot_cocomac_region_values

## Configuration

In [ ]:
results_path = "results"
data_path    = "data"

device   = "cuda" if torch.cuda.is_available() else "cpu"
FONTSIZE = 14
plt.rcParams.update({"font.size": FONTSIZE})

all_covs = load_covariance_dict(f"{data_path}/covariance_matrices_gc.h5")

# ── Optimal n-plets (from R1_A_lopo results) ─────────────────────────────────
# Keys: dataset -> polarity -> list of region indices
optimal_nplets = {
    "MA":  {"c_gt_nr": [7, 14, 23, 26, 35, 55, 67],   # Omega_C > Omega_NR, order 7
            "nr_gt_c": [38, 40, 45, 81]},              # Omega_NR > Omega_C, order 4
    "DBS": {"c_gt_nr": [6, 8, 12, 18, 20, 22, 59, 61, 63],  # order 9
            "nr_gt_c": [3, 4, 10]},                    # order 3
}

# ── Brain-map orders (must match what R1_B computed) ─────────────────────────
brainmap_orders = {
    "MA":  {"c_gt_nr": 7, "nr_gt_c": 4},
    "DBS": {"c_gt_nr": 9, "nr_gt_c": 3},
}

# ── Display settings ─────────────────────────────────────────────────────────
selected_measure = "O"  # column name in the per-scan DataFrame

pretty_labels = {
    "MA_awake":           "Awake",
    "ts_selv2":           "2% Sevo",
    "ts_selv4":           "4% Sevo",
    "deep_propofol":      "Deep Ppfl",
    "moderate_propofol":  "Light Ppfl",
    "ketamine":           "Ketamine",
    "DBS_awake":          "Awake",
    "ts_on_5V":           "CT high",
    "ts_off":             "off",
    "ts_on_3V_control":   "VT low",
    "ts_on_5V_control":   "VT high",
    "ts_on_3V":           "CT low",
}
custom_order = {
    "MA":  ["MA_awake", "ts_selv2", "ts_selv4", "moderate_propofol", "deep_propofol", "ketamine"],
    "DBS": ["DBS_awake", "ts_on_5V", "ts_on_3V", "ts_on_5V_control", "ts_on_3V_control", "ts_off"],
}
macrostates = {
    "Omitted":       ["ts_on_3V"],
    "Conscious":     ["MA_awake", "DBS_awake", "ts_on_5V"],
    "Non-responsive":["ts_selv2", "ts_selv4", "moderate_propofol", "deep_propofol",
                      "ketamine", "ts_off", "ts_on_3V_control", "ts_on_5V_control"],
}
state_colors = {
    "MA_awake":          "#FF9900",
    "ts_selv2":          "#75EACC",
    "ts_selv4":          "#00A073",
    "moderate_propofol": "#CEB8F2",
    "deep_propofol":     "#9C7AD8",
    "ketamine":          "#B0B900",
    "DBS_awake":         "#FF9900",
    "ts_on_5V":          "#FFD166",
    "ts_off":            "#9C7AD8",
    "ts_on_3V_control":  "#109121",
    "ts_on_5V_control":  "#109121",
    "ts_on_3V":          "#757575",
}

---
## Part 1 — Violin plots

For each dataset and each optimal n-plet we compute $\Omega$ for every scan
and plot violin distributions grouped by condition.
Significance brackets show Bonferroni-corrected two-sided Mann–Whitney U tests
comparing each condition against the awake baseline.

In [ ]:
def lighten(color, amount=0.3):
    c     = np.array(mcolors.to_rgb(color))
    white = np.array([1.0, 1.0, 1.0])
    return mcolors.to_hex((1 - amount) * c + amount * white)

def bonferroni(p_dict):
    m = len(p_dict)
    return {state: min(p * m, 1.0) for state, p in p_dict.items()}

def p_to_stars(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

def compute_mwu(df, dataset, base_state, measure):
    """Two-sided MWU p-values: base_state vs every other state."""
    pvals = {}
    for state in custom_order[dataset]:
        if state == base_state:
            continue
        x = df.loc[df["state"] == base_state, measure].values
        y = df.loc[df["state"] == state,      measure].values
        if x.size and y.size:
            _, p = mannwhitneyu(x, y, alternative="two-sided")
            pvals[state] = p
    return pvals

def add_sig_bracket(ax, x1, x2, y, h, text, fontsize=11):
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=1.2, c="black")
    ax.text((x1 + x2) / 2, y + h, text, ha="center", va="bottom", fontsize=fontsize)

In [ ]:
# ── Compute Ω for all scans, for every optimal n-plet ────────────────────────
scan_data = {}  # {(dataset, polarity): DataFrame}

for dataset, polarities in optimal_nplets.items():
    for polarity, nplet in polarities.items():
        records = []
        idx_t   = torch.tensor(nplet, dtype=torch.long)

        for state_name, state_covs in all_covs[dataset].items():
            cov_tensor = torch.as_tensor(state_covs)  # (n_subj, N, N)
            measures   = nplets_measures(
                cov_tensor,
                nplets=[nplet],
                covmat_precomputed=True,
            )
            # Fisher-z mean FC
            cov_sub = cov_tensor.index_select(1, idx_t).index_select(2, idx_t)
            var     = torch.diagonal(cov_sub, dim1=-2, dim2=-1).clamp(min=1e-12)
            std     = var.sqrt()
            denom   = std.unsqueeze(-1) * std.unsqueeze(-2)
            corr    = (cov_sub / denom.clamp(min=1e-12)).clamp(-0.999999, 0.999999)
            k       = idx_t.numel()
            mask    = torch.triu(torch.ones(k, k, dtype=torch.bool), diagonal=1)
            fc_z    = torch.atanh(corr[:, mask]).mean(dim=1).cpu().numpy()

            for subj in range(measures.shape[1]):
                records.append({
                    "state":    state_name,
                    "subject":  subj,
                    "O":        measures[0, subj, 2].item(),
                    "FC_mean_z": fc_z[subj],
                })

        scan_data[(dataset, polarity)] = pd.DataFrame(records)

print("Computed Ω for:", list(scan_data.keys()))

In [ ]:
# ── Plot: one 1×2 figure per dataset (left=c_gt_nr, right=nr_gt_c) ───────────
pval_records = []

for dataset in ["MA", "DBS"]:
    fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=False)
    sns.set_style("white")

    for ax, polarity in zip(axes, ["nr_gt_c", "c_gt_nr"]):
        df      = scan_data[(dataset, polarity)]
        palette = {s: state_colors[s] for s in custom_order[dataset] if s in df["state"].values}
        order   = [s for s in custom_order[dataset] if s in df["state"].values]
        base    = f"{dataset}_awake"

        sns.violinplot(
            data=df, x="state", y=selected_measure, hue="state",
            legend=False, inner="box", palette=palette, dodge=False,
            alpha=0.8, cut=0, order=order, linewidth=1.2, ax=ax,
        )
        ax.axhline(0, color="black", linestyle="--", zorder=1)
        ax.set_xticklabels(
            [pretty_labels[s] for s in order], rotation=45, ha="right"
        )
        ax.set_xlabel("")
        ax.set_ylabel(r"$\Omega$")
        polarity_label = (r"$\Omega_C > \Omega_{NR}$" if polarity == "c_gt_nr"
                          else r"$\Omega_{NR} > \Omega_C$")
        ax.set_title(f"{dataset} — {polarity_label}")
        sns.despine(ax=ax)

        # significance brackets
        p_raw = compute_mwu(df, dataset, base, selected_measure)
        p_adj = bonferroni(p_raw)
        for state, p in p_raw.items():
            pval_records.append({"dataset": dataset, "state": state,
                                  "polarity": polarity, "p": p, "p_adj": p_adj[state]})

        y_min = df[selected_measure].min()
        y_max = df[selected_measure].max()
        margin = 0.10 * (y_max - y_min)
        step   = 0.08 * (y_max - y_min)
        pos    = {s: i for i, s in enumerate(order)}
        level  = 0
        for state in order:
            if state == base or state not in p_adj:
                continue
            stars = p_to_stars(p_adj[state])
            if stars == "ns":
                continue
            y = y_max + margin + level * step
            add_sig_bracket(ax, pos[base], pos[state], y, 0.5 * step, stars)
            level += 1
        if level:
            ax.set_ylim(bottom=y_min - margin, top=y_max + margin + level * step * 1.2)

    plt.suptitle(f"Dataset: {dataset}", fontsize=FONTSIZE + 1)
    plt.tight_layout()
    plt.savefig(f"results/fig2_violins_{dataset}.pdf", bbox_inches="tight")
    plt.show()

pval_df = pd.DataFrame(pval_records)
print(pval_df)

---
## Part 2 — Regional participation brain maps

Loads the region participation maps produced by `R1_B_region_sampling.ipynb`
and renders one brain map per panel at the corresponding optimal order.

In [ ]:
maps_path = f"{results_path}/R1_C_region_maps_PRAUC_deltaO.pkl.gz"
with gzip.open(maps_path, "rb") as fh:
    region_maps = pickle.load(fh)
# structure: region_maps[dataset][order][polarity]["region_counts_percent"]
print("Loaded region maps.")

In [ ]:
# ── Fig 2 brain maps: one per (dataset × polarity), at optimal order ─────────
cmap_by_polarity = {"c_gt_nr": "Blues", "nr_gt_c": "Reds"}

for dataset in ["MA", "DBS"]:
    for polarity in ["c_gt_nr", "nr_gt_c"]:
        order  = brainmap_orders[dataset][polarity]
        z_map  = region_maps[dataset][order][polarity]["region_counts_percent"]
        polarity_label = (r"Ω_C > Ω_NR" if polarity == "c_gt_nr" else r"Ω_NR > Ω_C")

        print(f"\n{dataset} | {polarity_label} | order {order}")
        fig = plot_cocomac_region_values(
            z_map,
            cmap_name=cmap_by_polarity[polarity],
            colorbar_title="% of n-plets",
            fontsize=22,
            height=450,
            width=900,
            distance=0.73,
        )
        fig.show()